# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides guidance for loading, inspecting, and analyzing a FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata (as an object, not a list or dict)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published on: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review available Record Sets, Fields, and their IDs.

**Note:** All entities are referenced by their `@id`.

We list out available record sets and their fields using the Croissant schema structure.

In [ ]:
# List all record sets and fields with their @id
record_sets = dataset.record_sets

print("Available record sets and their fields:\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']} (name: {rs['name']})")
    print("Fields:")
    for field in rs['fields']:
        print(f"  - Field @id: {field['@id']} (name: {field['name']}, type: {field.get('dataType')})")
    print("")

### Example: Preview records using the record set `@id`

Let's preview records for one of the available record sets using the mlcroissant API.

In [ ]:
# Select the first record set for demonstration
if len(record_sets) > 0:
    example_record_set_id = record_sets[0]['@id']
else:
    example_record_set_id = None

if example_record_set_id is not None:
    print(f"Previewing records for record set @id: {example_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i >= 2:
            break # Just preview 3 records
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis.

All extraction uses `@id` references for record sets and fields.

In [ ]:
# Build list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

# Extract all data into pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    # Collect all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows for record set {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(3))
    else:
        print(f"No records found for record set {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping using specific field `@id` references.

**For demonstration,** we choose a numeric field from one record set and perform basic EDA.

In [ ]:
# Choose a record set and numeric field
chosen_record_set_id = None
numeric_field_id = None
group_field_id = None

# Try to find a record set with a numeric field (e.g., GAD-7/PHQ-9 score)
for rs in record_sets:
    for field in rs['fields']:
        if field.get('dataType') in ['schema:Float', 'schema:Integer', 'schema:Number']:
            # Example: field @id may contain 'gad_7_score', 'phq_9_score', or 'psq_score'
            numeric_field_id = field['@id']
            chosen_record_set_id = rs['@id']
            break
    if numeric_field_id is not None:
        break

# Attempt to select a group field (e.g., gender or education)
if chosen_record_set_id:
    for field in rs['fields']:
        if field.get('dataType') == 'schema:Text':
            if 'gender' in field['@id'].lower() or 'level_of_education' in field['@id'].lower():
                group_field_id = field['@id']
                break

# Now perform EDA on this record set and field
if chosen_record_set_id and numeric_field_id:
    df = dataframes[chosen_record_set_id]
    # Ensure the field is present
    if numeric_field_id in df.columns:
        # Remove outliers: filter scores > threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        
        # Normalize the numeric field
        mean_value = filtered_df[numeric_field_id].mean()
        std_value = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_value) / std_value
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Grouping
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Numeric field {numeric_field_id} not found in columns.")
else:
    print("No suitable numeric field found in the dataset. Please review the listed fields above.")

## 5. Visualization
Visualize data distributions or relationships between fields using standard plotting libraries.
You can reference column names by their `@id`.

In [ ]:
# Visualize numeric field distribution and group comparison
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {chosen_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable numeric field or group field found for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, inspect, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

We:
- Loaded metadata and explored the schema, referencing all entities by their `@id`.
- Inspected available record sets and fields.
- Extracted tabular data for analysis.
- Performed EDA, including filtering, normalization, and grouping by key demographic attributes.
- Visualized distributions and group comparisons.

Explore further by using additional fields, filter criteria, and more advanced visualizations!
